# Lenguajes y Autómatas II (SCD-1016)
## Sesión 06 · Sistema de Tipos Primitivos y Verificación Semántica

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo decide el compilador si una operación entre dos tipos tiene sentido?
>
> **Objetivo de la sesión:** Aplicar una tabla de compatibilidad de tipos y explicar por qué asignar un tipo más ancho a uno más angosto exige una conversión explícita.

**Entregable:** tu tabla en artesano_sesion06_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Implementas compatible(destino, origen) y nombre_tipo(t); el harness prueba asignaciones legales e ilegales y el nombre de cada tipo.

Al final de la sesión tendrás: **Una tabla de compatibilidad en C que acepta asignaciones seguras, rechaza las que pierden datos y nombra cada tipo.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 06](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion06.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion06.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion06.h"

int compatible(Tipo destino, Tipo origen);
const char *nombre_tipo(Tipo t);
```

- La jerarquía es CHAR < INT < FLOAT.
- Asignar a un tipo más angosto se rechaza: requiere cast explícito.
- TIPO_ERROR nunca es compatible con nada.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion06_c.c` junto con `artesano_sesion06_c.c`
> (o `ia_sesion06_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `compatible(INT,INT)` | 1 |
| 2 | `compatible(FLOAT,INT)` | 1 |
| 3 | `compatible(INT,FLOAT)` | 0 |
| 4 | `compatible(INT,ERROR)` | 0 |
| 5 | `nombre_tipo(INT)` | `int` |
| 6 | `nombre_tipo(ERROR)` | `error` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===
Caso                               esperado:              obtenido:              resultado
[ 1] compatible(INT,INT)                esperado: 1            obtenido: 1            OK
[ 2] compatible(FLOAT,INT)              esperado: 1            obtenido: 1            OK
[ 3] compatible(INT,FLOAT)              esperado: 0            obtenido: 0            OK
[ 4] compatible(INT,ERROR)              esperado: 0            obtenido: 0            OK
[ 5] nombre_tipo(INT)                   esperado: int          obtenido: int          OK
[ 6] nombre_tipo(ERROR)                 esperado: error        obtenido: error        OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] La compatibilidad solo acepta destinos de igual o mayor rango.
- [ ] TIPO_ERROR se rechaza siempre y se nombra como error.
- [ ] Los seis casos del harness salen OK con código de salida 0.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion06_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion06.h
/* api_sesion06.h - contrato de la Sesion 06
 * Tabla de compatibilidad de tipos
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION06_H
#define API_SESION06_H

typedef enum { TIPO_ERROR = 0, TIPO_CHAR = 1, TIPO_INT = 2, TIPO_FLOAT = 3 } Tipo;

int compatible(Tipo destino, Tipo origen);
const char *nombre_tipo(Tipo t);

#endif /* API_SESION06_H */


Writing api_sesion06.h


In [3]:
%%writefile artesano_sesion06_c.c
/* artesano_sesion06_c.c - Fase 1: El Artesano (sin IA)
 * Tabla de compatibilidad de tipos.
 */
#include "api_sesion06.h"
#include <stdio.h>
#include <string.h>

int compatible(Tipo destino, Tipo origen) {
    /* el error nunca es compatible con nada */
    if (destino == TIPO_ERROR || origen == TIPO_ERROR) {
        return 0;
    }
    /* jerarquia CHAR < INT < FLOAT: solo acepto destinos de igual o mayor rango */
    if (destino >= origen) {
        return 1;
    }
    return 0;  /* tipo mas angosto: necesita cast explicito */
}

const char *nombre_tipo(Tipo t) {
    switch (t) {
        case TIPO_CHAR:  return "char";
        case TIPO_INT:   return "int";
        case TIPO_FLOAT: return "float";
        case TIPO_ERROR: return "error";
        default:         return "desconocido";
    }
}

Writing artesano_sesion06_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion06_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


CONTEXTO: Estoy en la materia Lenguajes y Autómatas II, programando en C (estándar C11).
Estoy haciendo la verificación semántica de un compilador. El enum Tipo está definido en
api_sesion06.h: TIPO_ERROR = 0, TIPO_CHAR = 1, TIPO_INT = 2, TIPO_FLOAT = 3. Se compila
con gcc -std=c11 -Wall -Wextra -Werror, o sea que cualquier advertencia es error.

ROL: Actúa como ingeniero de software de sistemas y compiladores, con énfasis en sistemas
de tipos y verificación semántica.

ESPECIFICACIÓN: Implementa exactamente estas firmas, sin cambiarlas:
  int compatible(Tipo destino, Tipo origen);
  const char *nombre_tipo(Tipo t);
Reglas: la jerarquía es CHAR < INT < FLOAT; compatible devuelve 1 si el destino es de igual
o mayor rango que el origen y 0 si no (asignar a un tipo más angosto requiere cast
explícito); TIPO_ERROR nunca es compatible con nada, ni como destino ni como origen;
nombre_tipo devuelve "char", "int", "float" o "error".

OUTPUT: Entrégame solo el archivo .c con #include "api_sesion06.h", sin main(), con
comentarios breves y el análisis de complejidad de cada función.

In [4]:
%%writefile ia_sesion06_c.c
/* ia_sesion06_c.c - Fase 2: El Cientifico */
#include "api_sesion06.h"

int compatible(Tipo destino, Tipo origen) {
    if (destino == TIPO_ERROR || origen == TIPO_ERROR) {
        return 0;
    }
    return destino >= origen;
}

const char *nombre_tipo(Tipo t) {
    static const char *nombres[] = {"error", "char", "int", "float"};
    return nombres[t];
}


Writing ia_sesion06_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion06_c.c
/* harness_sesion06_c.c - auditoria automatica de la Sesion 06
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion06_c.c artesano_sesion06_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion06.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    {  verificar_int("compatible(INT,INT)", (long)(1), (long)(compatible(TIPO_INT, TIPO_INT))); }
    {  verificar_int("compatible(FLOAT,INT)", (long)(1), (long)(compatible(TIPO_FLOAT, TIPO_INT))); }
    {  verificar_int("compatible(INT,FLOAT)", (long)(0), (long)(compatible(TIPO_INT, TIPO_FLOAT))); }
    {  verificar_int("compatible(INT,ERROR)", (long)(0), (long)(compatible(TIPO_INT, TIPO_ERROR))); }
    {  verificar_str("nombre_tipo(INT)", "int", nombre_tipo(TIPO_INT)); }
    {  verificar_str("nombre_tipo(ERROR)", "error", nombre_tipo(TIPO_ERROR)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md" "\n");
    return 0;
}


Writing harness_sesion06_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion06_c.c artesano_sesion06_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===
Caso                               esperado:              obtenido:              resultado
[ 1] compatible(INT,INT)                esperado: 1            obtenido: 1            OK
[ 2] compatible(FLOAT,INT)              esperado: 1            obtenido: 1            OK
[ 3] compatible(INT,FLOAT)              esperado: 0            obtenido: 0            OK
[ 4] compatible(INT,ERROR)              esperado: 0            obtenido: 0            OK
[ 5] nombre_tipo(INT)                   esperado: int          obtenido: int          OK
[ 6] nombre_tipo(ERROR)                 esperado: error        obtenido: error        OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion06_c.c ia_sesion06_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 06 - Tabla de compatibilidad de tipos ===
Caso                               esperado:              obtenido:              resultado
[ 1] compatible(INT,INT)                esperado: 1            obtenido: 1            OK
[ 2] compatible(FLOAT,INT)              esperado: 1            obtenido: 1            OK
[ 3] compatible(INT,FLOAT)              esperado: 0            obtenido: 0            OK
[ 4] compatible(INT,ERROR)              esperado: 0            obtenido: 0            OK
[ 5] nombre_tipo(INT)                   esperado: int          obtenido: int          OK
[ 6] nombre_tipo(ERROR)                 esperado: error        obtenido: error        OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion06.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion06.md` a `auditorias/auditoria_sesion06.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
